# Homogeneous Propagation Medium Example

Ported from: k-Wave/examples/example_ivp_homogeneous_medium.m

Simulates the pressure field generated by an initial pressure distribution
(two discs of different magnitudes) within a two-dimensional homogeneous
propagation medium with power-law absorption. The acoustic field is detected
using a centered Cartesian circular sensor.

This is the simplest 2D k-Wave example and serves as a starting point for
the other initial-value-problem demonstrations. You should observe two
expanding circular wavefronts (one from each disc) that are progressively
attenuated by the absorbing medium.

In [ ]:
!pip install k-wave-python

In [ ]:
import numpy as np

from kwave.data import Vector
from kwave.kgrid import kWaveGrid
from kwave.kmedium import kWaveMedium
from kwave.ksensor import kSensor
from kwave.ksource import kSource
from kwave.kspaceFirstOrder import kspaceFirstOrder
from kwave.utils.mapgen import make_cart_circle, make_disc

In [ ]:
def setup():
    """Set up the simulation physics (grid, medium, source).

    Returns:
        tuple: (kgrid, medium, source)
    """

    # create the computational grid
    Nx = 128  # number of grid points in the x direction
    Ny = 128  # number of grid points in the y direction
    dx = 0.1e-3  # grid point spacing in the x direction [m]
    dy = 0.1e-3  # grid point spacing in the y direction [m]
    kgrid = kWaveGrid(Vector([Nx, Ny]), Vector([dx, dy]))

    # define the properties of the propagation medium
    medium = kWaveMedium(
        sound_speed=1500,  # [m/s]
        alpha_coeff=0.75,  # [dB/(MHz^y cm)]
        alpha_power=1.5,
    )

    # create initial pressure distribution using make_disc
    # -- disc 1 --
    disc_1_magnitude = 5  # [Pa]
    disc_1_x_pos = 50  # [grid points, 1-based]
    disc_1_y_pos = 50  # [grid points, 1-based]
    disc_1_radius = 8  # [grid points]
    disc_1 = disc_1_magnitude * make_disc(Vector([Nx, Ny]), Vector([disc_1_x_pos, disc_1_y_pos]), disc_1_radius)

    # -- disc 2 --
    disc_2_magnitude = 3  # [Pa]
    disc_2_x_pos = 80  # [grid points, 1-based]
    disc_2_y_pos = 60  # [grid points, 1-based]
    disc_2_radius = 5  # [grid points]
    disc_2 = disc_2_magnitude * make_disc(Vector([Nx, Ny]), Vector([disc_2_x_pos, disc_2_y_pos]), disc_2_radius)

    source = kSource()
    source.p0 = (disc_1 + disc_2).astype(float)

    # set time array
    kgrid.makeTime(medium.sound_speed)

    return kgrid, medium, source

In [ ]:
def run(backend="python", device="cpu", quiet=True):
    """Run with the original Cartesian circular sensor.

    Returns:
        dict: Simulation results with key 'p' (sensor_points x time_steps).
    """
    kgrid, medium, source = setup()

    # define a centered circular sensor (Cartesian)
    sensor_radius = 4e-3  # [m]
    num_sensor_points = 50
    sensor_mask = make_cart_circle(sensor_radius, num_sensor_points)
    sensor = kSensor(mask=sensor_mask)

    return kspaceFirstOrder(
        kgrid,
        medium,
        source,
        sensor,
        backend=backend,
        device=device,
        quiet=quiet,
        pml_inside=True,
    )

In [ ]:
if __name__ == "__main__":
    import matplotlib.pyplot as plt

    result = run(quiet=False)
    p = np.asarray(result["p"])

    fig, axes = plt.subplots(1, 2, figsize=(12, 5))

    # plot the simulated sensor data
    ax = axes[0]
    im = ax.imshow(p, aspect="auto", vmin=-1, vmax=1)
    ax.set_ylabel("Sensor Position")
    ax.set_xlabel("Time Step")
    ax.set_title("Sensor Data")
    fig.colorbar(im, ax=ax)

    # plot the initial pressure distribution with sensor overlay
    kgrid, _, source = setup()
    sensor_mask = make_cart_circle(4e-3, 50)
    Nx, Ny = 128, 128
    ax = axes[1]
    ax.imshow(
        source.p0.T,
        extent=[
            kgrid.x_vec[0] * 1e3,
            kgrid.x_vec[-1] * 1e3,
            kgrid.y_vec[-1] * 1e3,
            kgrid.y_vec[0] * 1e3,
        ],
        vmin=-1,
        vmax=1,
        cmap="RdBu_r",
    )
    ax.plot(sensor_mask[0, :] * 1e3, sensor_mask[1, :] * 1e3, "k.", markersize=3)
    ax.set_xlabel("y-position [mm]")
    ax.set_ylabel("x-position [mm]")
    ax.set_title("Initial Pressure + Sensor")
    ax.set_aspect("equal")

    fig.suptitle("Homogeneous Propagation Medium Example")
    fig.tight_layout()
    plt.show()